# VAE x64 Baseline

This notebook is the curated submission-facing entry point for the convolutional VAE baseline on the shared `x64` anomaly benchmark.
It defaults to reusing the saved checkpoint and evaluation artifacts instead of retraining.

## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/vae/x64/baseline/train_config.toml`
- Training script: `scripts/train_vae.py`
- Artifact root: `experiments/anomaly_detection/vae/x64/baseline/artifacts/vae_baseline/`
- Default behavior: reuse `best_model.pt`, `history.json`, and `results/evaluation/` outputs when they already exist

### Setup And Imports

This cell resolves the repo root, exposes `src/` on `PYTHONPATH`, and imports the shared dataset loader plus the VAE model used for qualitative reconstructions.

In [ ]:
from __future__ import annotations
import json
import os
import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import torch
try:
    from IPython.display import display
except ImportError:

    def display(obj: object) -> None:
        print(obj)
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'wafer_defect').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.models.vae import ConvVariationalAutoencoder, VAEOutput

### Load The Local Config And Run Controls

This cell points the notebook at the local baseline config and exposes the main rerun flags. Leave them as `False` for the normal artifact-first submission flow.

In [ ]:
try:
    CONFIG_PATH = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'baseline' / 'train_config.toml'
    config = load_toml(CONFIG_PATH)
    OUTPUT_DIR = REPO_ROOT / config['run']['output_dir']
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    EVALUATION_DIR = OUTPUT_DIR / 'results/evaluation'
    PLOTS_DIR = OUTPUT_DIR / 'plots'
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    FORCE_RETRAIN = False
    FORCE_EVALUATION_RERUN = False
    display(config)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "config_path = repo_root / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'baseline' / 'train_config.toml'\nconfig = load_toml(config_path)\noutput_dir = repo_root / config['run']['output_dir']\noutput_dir.mkdir(parents=true, exist_ok=true)\nevaluation_dir = output_dir / 'results/evaluation'\nplots_dir = output_dir / 'plots'\nplots_dir.mkdir(parents=true, exist_ok=true)\nforce_retrain = false\nforce_evaluation_rerun = false\ndisplay(config)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Preview The Curated Benchmark Split

This cell confirms that the notebook is reading the processed `50k / 5%` benchmark metadata and shows the split counts.

In [ ]:
try:
    metadata_path = REPO_ROOT / config['data']['metadata_csv']
    metadata_df = pd.read_csv(metadata_path)
    split_summary_df = metadata_df.groupby(['split', 'is_anomaly']).size().reset_index(name='count').sort_values(['split', 'is_anomaly']).reset_index(drop=True)
    test_dataset = WaferMapDataset(metadata_path, split='test', image_size=int(config['data']['image_size']))
    print(f'Metadata CSV: {metadata_path}')
    display(split_summary_df)
    display(metadata_df.head(5))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "metadata_path = repo_root / config['data']['metadata_csv']\nmetadata_df = pd.read_csv(metadata_path)\nsplit_summary_df = metadata_df.groupby(['split', 'is_anomaly']).size().reset_index(name='count').sort_values(['split', 'is_anomaly']).reset_index(drop=true)\ntest_dataset = wafermapdataset(metadata_path, split='test', image_size=int(config['data']['image_size']))\nprint(f'metadata csv: {metadata_path}')\ndisplay(split_summary_df)\ndisplay(metadata_df.head(5))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Train Or Reuse Saved Artifacts

This cell either reuses the saved VAE artifacts or runs `scripts/train_vae.py` with the local config. The default path is to load the existing checkpoint and history files.

In [ ]:
try:
    history_path = OUTPUT_DIR / 'results' / 'history.json'
    summary_path = OUTPUT_DIR / 'results' / 'summary.json'
    best_model_path = OUTPUT_DIR / 'checkpoints' / 'best_model.pt'
    artifacts_ready = history_path.exists() and summary_path.exists() and best_model_path.exists()
    env = os.environ.copy()
    env['PYTHONPATH'] = str(SRC_ROOT) if not env.get('PYTHONPATH') else str(SRC_ROOT) + os.pathsep + env['PYTHONPATH']
    if FORCE_RETRAIN:
        train_cmd = [sys.executable, 'scripts/train_vae.py', '--config', str(CONFIG_PATH.relative_to(REPO_ROOT))]
        print('Running:', ' '.join(train_cmd))
        subprocess.run(train_cmd, cwd=REPO_ROOT, env=env, check=True)
    elif artifacts_ready:
        print(f'Found existing training artifacts in {OUTPUT_DIR}. Skipping retraining.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    history = json.loads(history_path.read_text(encoding='utf-8'))
    history_df = pd.DataFrame(history)
    training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([training_summary]))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "history_path = output_dir / 'results' / 'history.json'\nsummary_path = output_dir / 'results' / 'summary.json'\nbest_model_path = output_dir / 'checkpoints' / 'best_model.pt'\nartifacts_ready = history_path.exists() and summary_path.exists() and best_model_path.exists()\nenv = os.environ.copy()\nenv['pythonpath'] = str(src_root) if not env.get('pythonpath') else str(src_root) + os.pathsep + env['pythonpath']\nif force_retrain:\n    train_cmd = [sys.executable, 'scripts/train_vae.py', '--config', str(config_path.relative_to(repo_root))]\n    print('running:', ' '.join(train_cmd))\n    subprocess.run(train_cmd, cwd=repo_root, env=env, check=true)\nelif artifacts_ready:\n    print(f'found existing training artifacts in {output_dir}. skipping retraining.')\nelse:\n    print('[warning] the rerun/training flags are false and the saved artifacts for this section are missing. skipping this section.')\nhistory = json.loads(history_path.read_text(encoding='utf-8'))\nhistory_df = pd.dataframe(history)\ntraining_summary = json.loads(summary_path.read_text(encoding='utf-8'))\ndisplay(pd.dataframe([training_summary]))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Plot The Training Curves

This cell recreates the total, reconstruction, and KL-loss curves from `history.json`, shows them inline, and saves them to the local `plots/` folder.

In [ ]:
try:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
    axes[0].set_title('Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history_df['epoch'], history_df['train_reconstruction_loss'], label='train')
    axes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label='val')
    axes[1].set_title('Reconstruction Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[2].plot(history_df['epoch'], history_df['train_kl_loss'], label='train')
    axes[2].plot(history_df['epoch'], history_df['val_kl_loss'], label='val')
    axes[2].set_title('KL Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    fig.savefig(PLOTS_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=true)\naxes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')\naxes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')\naxes[0].set_title('total loss')\naxes[0].set_xlabel('epoch')\naxes[0].legend()\naxes[1].plot(history_df['epoch'], history_df['train_reconstruction_loss'], label='train')\naxes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label='val')\naxes[1].set_title('reconstruction loss')\naxes[1].set_xlabel('epoch')\naxes[1].legend()\naxes[2].plot(history_df['epoch'], history_df['train_kl_loss'], label='train')\naxes[2].plot(history_df['epoch'], history_df['val_kl_loss'], label='val')\naxes[2].set_title('kl loss')\naxes[2].set_xlabel('epoch')\naxes[2].legend()\nfig.savefig(plots_dir / 'training_curves.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Evaluate Or Reuse Saved Scores

This cell reuses the saved evaluation CSVs and summary when they already exist. If the evaluation cache is missing, it reruns the shared reconstruction-evaluation script against the saved checkpoint.

In [ ]:
try:
    evaluation_summary_path = EVALUATION_DIR / 'summary.json'
    val_scores_path = EVALUATION_DIR / 'val_scores.csv'
    test_scores_path = EVALUATION_DIR / 'test_scores.csv'
    threshold_sweep_path = EVALUATION_DIR / 'threshold_sweep.csv'
    evaluation_ready = all((path.exists() for path in [evaluation_summary_path, val_scores_path, test_scores_path, threshold_sweep_path]))
    if FORCE_EVALUATION_RERUN:
        eval_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str(best_model_path.relative_to(REPO_ROOT)), '--config', str(CONFIG_PATH.relative_to(REPO_ROOT)), '--output-dir', str(EVALUATION_DIR.relative_to(REPO_ROOT))]
        print('Running:', ' '.join(eval_cmd))
        subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, check=True)
    elif evaluation_ready:
        print(f'Found existing evaluation outputs in {EVALUATION_DIR}. Skipping rerun.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    evaluation_summary = json.loads(evaluation_summary_path.read_text(encoding='utf-8'))
    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)
    threshold = float(evaluation_summary['threshold'])
    best_sweep = dict(evaluation_summary.get('best_threshold_sweep', {}))
    display(pd.DataFrame([evaluation_summary['metrics_at_validation_threshold']]))
    display(pd.DataFrame([best_sweep]))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "evaluation_summary_path = evaluation_dir / 'summary.json'\nval_scores_path = evaluation_dir / 'val_scores.csv'\ntest_scores_path = evaluation_dir / 'test_scores.csv'\nthreshold_sweep_path = evaluation_dir / 'threshold_sweep.csv'\nevaluation_ready = all((path.exists() for path in [evaluation_summary_path, val_scores_path, test_scores_path, threshold_sweep_path]))\nif force_evaluation_rerun:\n    eval_cmd = [sys.executable, 'scripts/evaluate_reconstruction_model.py', '--checkpoint', str(best_model_path.relative_to(repo_root)), '--config', str(config_path.relative_to(repo_root)), '--output-dir', str(evaluation_dir.relative_to(repo_root))]\n    print('running:', ' '.join(eval_cmd))\n    subprocess.run(eval_cmd, cwd=repo_root, env=env, check=true)\nelif evaluation_ready:\n    print(f'found existing evaluation outputs in {evaluation_dir}. skipping rerun.')\nelse:\n    print('[warning] the rerun/training flags are false and the saved artifacts for this section are missing. skipping this section.')\nevaluation_summary = json.loads(evaluation_summary_path.read_text(encoding='utf-8'))\nval_scores_df = pd.read_csv(val_scores_path)\ntest_scores_df = pd.read_csv(test_scores_path)\nthreshold_sweep_df = pd.read_csv(threshold_sweep_path)\nthreshold = float(evaluation_summary['threshold'])\nbest_sweep = dict(evaluation_summary.get('best_threshold_sweep', {}))\ndisplay(pd.dataframe([evaluation_summary['metrics_at_validation_threshold']]))\ndisplay(pd.dataframe([best_sweep]))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Plot Score Distributions And Threshold Sweep

This cell recreates the main score and threshold-sweep plots from the saved evaluation outputs and writes them to the local artifact folder.

In [ ]:
try:
    cm = evaluation_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])
    cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
    axes[0].hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')
    axes[0].hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')
    axes[0].axvline(threshold, color='red', linestyle='--', label=f'val threshold = {threshold:.4f}')
    axes[0].set_title('Test Score Distribution')
    axes[0].set_xlabel('Anomaly score')
    axes[0].legend()
    axes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    axes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    axes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    axes[1].axvline(threshold, color='red', linestyle='--', label='val threshold')
    axes[1].axvline(float(best_sweep['threshold']), color='green', linestyle=':', label='best sweep')
    axes[1].set_title('Threshold Sweep')
    axes[1].set_xlabel('Threshold')
    axes[1].legend()
    heatmap = axes[2].imshow(cm_df.to_numpy(), cmap='Blues')
    axes[2].set_xticks(range(cm_df.shape[1]), cm_df.columns)
    axes[2].set_yticks(range(cm_df.shape[0]), cm_df.index)
    axes[2].set_title('Confusion Matrix')
    axes[2].set_xlabel('Predicted label')
    axes[2].set_ylabel('True label')
    for row_idx in range(cm_df.shape[0]):
        for col_idx in range(cm_df.shape[1]):
            axes[2].text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black')
    fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
    fig.savefig(PLOTS_DIR / 'score_distribution_sweep_confusion.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    display(cm_df)
    display(threshold_sweep_df.sort_values('f1', ascending=False).head(10))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "cm = evaluation_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])\ncm_df = pd.dataframe(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])\nfig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=true)\naxes[0].hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')\naxes[0].hist(test_scores_df.loc[test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')\naxes[0].axvline(threshold, color='red', linestyle='--', label=f'val threshold = {threshold:.4f}')\naxes[0].set_title('test score distribution')\naxes[0].set_xlabel('anomaly score')\naxes[0].legend()\naxes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')\naxes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')\naxes[1].plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')\naxes[1].axvline(threshold, color='red', linestyle='--', label='val threshold')\naxes[1].axvline(float(best_sweep['threshold']), color='green', linestyle=':', label='best sweep')\naxes[1].set_title('threshold sweep')\naxes[1].set_xlabel('threshold')\naxes[1].legend()\nheatmap = axes[2].imshow(cm_df.to_numpy(), cmap='blues')\naxes[2].set_xticks(range(cm_df.shape[1]), cm_df.columns)\naxes[2].set_yticks(range(cm_df.shape[0]), cm_df.index)\naxes[2].set_title('confusion matrix')\naxes[2].set_xlabel('predicted label')\naxes[2].set_ylabel('true label')\nfor row_idx in range(cm_df.shape[0]):\n    for col_idx in range(cm_df.shape[1]):\n        axes[2].text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black')\nfig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)\nfig.savefig(plots_dir / 'score_distribution_sweep_confusion.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)\ndisplay(cm_df)\ndisplay(threshold_sweep_df.sort_values('f1', ascending=false).head(10))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


### Show Reconstruction Examples

This cell loads the saved `best_model.pt` checkpoint, reconstructs a small test batch, and saves the qualitative comparison figure alongside the other plots.

In [ ]:
try:
    checkpoint = torch.load(best_model_path, map_location='cpu')
    eval_model = ConvVariationalAutoencoder(latent_dim=int(config['model']['latent_dim']), image_size=int(config['data']['image_size']))
    eval_model.load_state_dict(checkpoint['model_state_dict'])
    eval_model.eval()
    sample_count = min(4, len(test_dataset))
    sample_tensors = []
    sample_labels = []
    for index in range(sample_count):
        tensor, label = test_dataset[index]
        sample_tensors.append(tensor)
        sample_labels.append(int(label))
    batch = torch.stack(sample_tensors, dim=0)
    with torch.inference_mode():
        outputs = eval_model(batch)
    if not isinstance(outputs, VAEOutput):
        raise TypeError('VAE model must return VAEOutput')
    recons = outputs.reconstruction.cpu()
    fig, axes = plt.subplots(sample_count, 4, figsize=(9, 2.5 * sample_count), constrained_layout=True)
    if sample_count == 1:
        axes = [axes]
    for row in range(sample_count):
        axes[row][0].imshow(batch[row, 0], cmap='gray')
        axes[row][0].set_title(f'Input {row}')
        axes[row][1].imshow(recons[row, 0], cmap='gray')
        axes[row][1].set_title('Recon')
        axes[row][2].imshow((batch[row, 0] - recons[row, 0]).abs(), cmap='magma')
        axes[row][2].set_title('Abs Error')
        axes[row][3].text(0.05, 0.6, f'label={sample_labels[row]}', fontsize=11)
        axes[row][3].axis('off')
        for col in range(3):
            axes[row][col].set_xticks([])
            axes[row][col].set_yticks([])
    fig.savefig(PLOTS_DIR / 'reconstruction_examples.png', dpi=160, bbox_inches='tight')
    display(fig)
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "checkpoint = torch.load(best_model_path, map_location='cpu')\neval_model = convvariationalautoencoder(latent_dim=int(config['model']['latent_dim']), image_size=int(config['data']['image_size']))\neval_model.load_state_dict(checkpoint['model_state_dict'])\neval_model.eval()\nsample_count = min(4, len(test_dataset))\nsample_tensors = []\nsample_labels = []\nfor index in range(sample_count):\n    tensor, label = test_dataset[index]\n    sample_tensors.append(tensor)\n    sample_labels.append(int(label))\nbatch = torch.stack(sample_tensors, dim=0)\nwith torch.inference_mode():\n    outputs = eval_model(batch)\nif not isinstance(outputs, vaeoutput):\n    raise typeerror('vae model must return vaeoutput')\nrecons = outputs.reconstruction.cpu()\nfig, axes = plt.subplots(sample_count, 4, figsize=(9, 2.5 * sample_count), constrained_layout=true)\nif sample_count == 1:\n    axes = [axes]\nfor row in range(sample_count):\n    axes[row][0].imshow(batch[row, 0], cmap='gray')\n    axes[row][0].set_title(f'input {row}')\n    axes[row][1].imshow(recons[row, 0], cmap='gray')\n    axes[row][1].set_title('recon')\n    axes[row][2].imshow((batch[row, 0] - recons[row, 0]).abs(), cmap='magma')\n    axes[row][2].set_title('abs error')\n    axes[row][3].text(0.05, 0.6, f'label={sample_labels[row]}', fontsize=11)\n    axes[row][3].axis('off')\n    for col in range(3):\n        axes[row][col].set_xticks([])\n        axes[row][col].set_yticks([])\nfig.savefig(plots_dir / 'reconstruction_examples.png', dpi=160, bbox_inches='tight')\ndisplay(fig)\nplt.close(fig)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
